# CWRU Bearing 데이터 전처리 및 모델 성능 분석
**HybridPdM - 1D-CNN(WDCNN) 기반 베어링 결함 다중 분류 (raw vs STFT 비교)**

DE(Drive End) 가속도 신호 → Sliding Window(1024) → Wide-Kernel CNN → IR/B/OR 3-class

In [ ]:
import warnings, re
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

import scipy.io as sio
from scipy.signal import stft
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix, classification_report,
    f1_score, precision_score, recall_score, accuracy_score
)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 120
SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)

GREEN = '#3a9a5c'; RED = '#e05c5c'; BLUE = '#4a7fc1'; ORANGE = '#e8a838'; PURPLE = '#9b59b6'
CLASS_NAMES = ['IR (내륜)', 'B (볼)', 'OR (외륜)']
CLASS_COLORS = [BLUE, ORANGE, RED]

DATA_DIR = Path('../dataset/10987113')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'설정 완료. device={device}')

---
## 01 데이터 로드 및 클래스 매핑
CWRU 공식 카탈로그의 파일 번호 → 결함 유형 매핑을 사용한다. 라벨이 명확한 파일만 사용.

In [ ]:
def cwru_class_for(num):
    ir_ranges = [(105,108),(169,172),(209,212),(278,281)]
    b_ranges  = [(118,121),(185,188),(222,225),(282,285),(286,289)]
    or_ranges = [(130,133),(144,147),(156,159),(197,200),(234,237),
                 (246,249),(258,261),(294,297),(298,301),(310,313)]
    def _in(n, rngs): return any(a<=n<=b for a,b in rngs)
    if _in(num, ir_ranges): return 0
    if _in(num, b_ranges):  return 1
    if _in(num, or_ranges): return 2
    return None

def read_signal(fp):
    d = sio.loadmat(str(fp))
    de_keys = [k for k in d if k.endswith('_DE_time')]
    fe_keys = [k for k in d if k.endswith('_FE_time')]
    key = de_keys[0] if de_keys else (fe_keys[0] if fe_keys else None)
    if key is None: return None
    return np.asarray(d[key]).reshape(-1).astype(np.float32)

# 파일 수집
files = []
for f in sorted(DATA_DIR.glob('*.mat')):
    m = re.match(r'(\d+)\.mat', f.name)
    if not m: continue
    cls = cwru_class_for(int(m.group(1)))
    if cls is None: continue
    files.append((f, cls))

class_counts = np.bincount([c for _,c in files], minlength=3)
print(f'유효 파일 수: {len(files)}')
for i, name in enumerate(CLASS_NAMES):
    print(f'  {name}: {class_counts[i]}개 파일')

---
## 02 신호 시각화

### 2-1. 클래스별 raw 진동 신호 (시간 도메인)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 7), sharex=True)

samples_by_class = {0:None, 1:None, 2:None}
for fp, cls in files:
    if samples_by_class[cls] is None:
        sig = read_signal(fp)
        if sig is not None and len(sig) > 4096:
            samples_by_class[cls] = (fp.name, sig[:4096])
    if all(v is not None for v in samples_by_class.values()): break

for i in range(3):
    ax = axes[i]
    fname, sig = samples_by_class[i]
    ax.plot(sig, color=CLASS_COLORS[i], linewidth=0.6)
    ax.set_title(f'{CLASS_NAMES[i]} ({fname}) - 첫 4096 샘플',
                 fontweight='bold', fontsize=10)
    ax.set_ylabel('가속도')
    ax.grid(alpha=0.3)
axes[-1].set_xlabel('샘플 인덱스')
plt.suptitle('CWRU - 클래스별 진동 신호 (시간 도메인)', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

### 2-2. FFT 주파수 스펙트럼 (베어링 결함 주파수가 보임)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 7), sharex=True)
fs = 12000  # CWRU sampling rate (Hz)

for i in range(3):
    ax = axes[i]
    _, sig = samples_by_class[i]
    N = len(sig)
    freqs = np.fft.rfftfreq(N, d=1/fs)
    mag = np.abs(np.fft.rfft(sig))
    ax.plot(freqs, mag, color=CLASS_COLORS[i], linewidth=0.8)
    ax.set_title(f'{CLASS_NAMES[i]} - FFT 스펙트럼', fontweight='bold', fontsize=10)
    ax.set_ylabel('Magnitude')
    ax.set_xlim(0, fs/2)
    ax.grid(alpha=0.3)
axes[-1].set_xlabel('주파수 (Hz)')
plt.suptitle('CWRU - 클래스별 주파수 스펙트럼 (fs=12kHz)', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

---
## 03 슬라이딩 윈도우 + 파일 단위 분할
윈도우 단위로 random split하면 같은 .mat의 인접 윈도우가 train/test에 동시 들어가 leakage가 발생한다.
**파일을 먼저 stratified split** 후 각 split의 파일에서만 윈도우를 추출한다.

In [ ]:
WINDOW, STRIDE, MAX_PER_FILE = 1024, 512, 200

file_idx = np.arange(len(files))
file_cls = np.array([c for _,c in files])
strat = file_cls if (np.bincount(file_cls).min() >= 3) else None

idx_tv, idx_te = train_test_split(file_idx, test_size=0.15,
                                    random_state=SEED, stratify=strat)
strat_tv = file_cls[idx_tv] if strat is not None else None
idx_tr, idx_va = train_test_split(idx_tv, test_size=0.15/0.85,
                                    random_state=SEED, stratify=strat_tv)

def extract(indices):
    Xs, ys = [], []
    for i in indices:
        fp, cls = files[int(i)]
        sig = read_signal(fp)
        if sig is None or sig.size < WINDOW: continue
        n = min((sig.size - WINDOW) // STRIDE + 1, MAX_PER_FILE)
        for k in range(n):
            Xs.append(sig[k*STRIDE : k*STRIDE + WINDOW])
            ys.append(cls)
    X = np.stack(Xs).astype(np.float32)[:, None, :]
    y = np.array(ys, dtype=np.int64)
    # per-window z-score
    mu = X.mean(axis=-1, keepdims=True)
    sd = X.std(axis=-1, keepdims=True) + 1e-6
    return ((X - mu) / sd).astype(np.float32), y

X_tr, y_tr = extract(idx_tr)
X_va, y_va = extract(idx_va)
X_te, y_te = extract(idx_te)

print(f'Train: {X_tr.shape}  클래스 분포: {np.bincount(y_tr)}')
print(f'Val:   {X_va.shape}  클래스 분포: {np.bincount(y_va)}')
print(f'Test:  {X_te.shape}  클래스 분포: {np.bincount(y_te)}')

fig, ax = plt.subplots(figsize=(9, 4.5))
x_pos = np.arange(3); w = 0.25
for j, (name, y_) in enumerate(zip(['Train','Val','Test'], [y_tr, y_va, y_te])):
    counts = np.bincount(y_, minlength=3)
    ax.bar(x_pos + (j-1)*w, counts, w, label=name,
           color=[GREEN,BLUE,RED][j], alpha=0.85)
ax.set_xticks(x_pos); ax.set_xticklabels(CLASS_NAMES)
ax.set_title('파일 단위 분할 후 클래스별 윈도우 수', fontweight='bold')
ax.set_ylabel('윈도우 수'); ax.legend()
plt.tight_layout(); plt.show()

---
## 04 WDCNN1D (Wide-Kernel Deep CNN) 학습

In [ ]:
class WDCNN1D(nn.Module):
    def __init__(self, n_classes=3, dropout=0.3):
        super().__init__()
        self.n_classes = n_classes
        self.conv1 = nn.Conv1d(1, 16, kernel_size=64, stride=8, padding=30)
        self.bn1 = nn.BatchNorm1d(16)
        self.conv2 = nn.Conv1d(16, 32, 3, padding=1); self.bn2 = nn.BatchNorm1d(32)
        self.conv3 = nn.Conv1d(32, 64, 3, padding=1); self.bn3 = nn.BatchNorm1d(64)
        self.gap = nn.AdaptiveAvgPool1d(4)
        self.drop = nn.Dropout(dropout)
        self.fc1 = nn.Linear(64*4, 64); self.fc2 = nn.Linear(64, n_classes)
    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x))); x = F.max_pool1d(x, 2)
        x = F.relu(self.bn2(self.conv2(x))); x = F.max_pool1d(x, 2)
        x = F.relu(self.bn3(self.conv3(x)))
        x = self.gap(x).flatten(1)
        x = self.drop(F.relu(self.fc1(x)))
        return self.fc2(x)

def train_cls(model, X_tr, y_tr, X_va, y_va, epochs=30, lr=1e-3, bs=128, patience=8):
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    crit = nn.CrossEntropyLoss()
    loader = DataLoader(TensorDataset(torch.tensor(X_tr), torch.tensor(y_tr)),
                        batch_size=bs, shuffle=True)
    Xva_t = torch.tensor(X_va).to(device); yva_t = torch.tensor(y_va).to(device)
    tr_losses, va_losses, va_accs = [], [], []
    best_acc, best_st, cnt = 0.0, None, 0
    for ep in range(1, epochs+1):
        model.train(); el = []
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = crit(model(xb), yb); loss.backward(); opt.step()
            el.append(loss.item())
        model.eval()
        with torch.no_grad():
            out = model(Xva_t)
            vl = crit(out, yva_t).item()
            va = (out.argmax(1) == yva_t).float().mean().item()
        tr_losses.append(np.mean(el)); va_losses.append(vl); va_accs.append(va)
        if va > best_acc:
            best_acc, best_st, cnt = va, {k:v.clone() for k,v in model.state_dict().items()}, 0
        else:
            cnt += 1
            if cnt >= patience:
                print(f'Early stop at epoch {ep}'); break
        if ep % 5 == 0:
            print(f'Epoch {ep:3d} | train={np.mean(el):.4f} | val_loss={vl:.4f} | val_acc={va:.4f}')
    model.load_state_dict(best_st)
    return model, tr_losses, va_losses, va_accs

print('학습 시작 (raw 1D)...')
model_raw, tr_l, va_l, va_a = train_cls(WDCNN1D(n_classes=3), X_tr, y_tr, X_va, y_va, epochs=30)

---
## 05 학습 곡선 & raw 1D 성능 평가

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
ep_range = range(1, len(tr_l)+1)
axes[0].plot(ep_range, tr_l, color=GREEN, linewidth=2, label='Train Loss')
axes[0].plot(ep_range, va_l, color=RED, linewidth=2, linestyle='--', label='Val Loss')
axes[0].set_title('학습 곡선', fontweight='bold'); axes[0].legend()
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('CE Loss')

axes[1].plot(ep_range, va_a, color=BLUE, linewidth=2, marker='o')
axes[1].axhline(max(va_a), color='gold', linestyle='--', linewidth=1.5,
                label=f'Best Val Acc {max(va_a):.4f}')
axes[1].set_title('Validation Accuracy', fontweight='bold'); axes[1].legend()
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].set_ylim(0, 1.05)
plt.tight_layout(); plt.show()

# Test 평가
model_raw.eval()
with torch.no_grad():
    Xte_t = torch.tensor(X_te).to(device)
    logits_raw = model_raw(Xte_t).cpu().numpy()
y_pred_raw = logits_raw.argmax(1)

acc_raw = accuracy_score(y_te, y_pred_raw)
f1_raw  = f1_score(y_te, y_pred_raw, average='macro', zero_division=0)
print(f'\n[raw 1D Test] Accuracy={acc_raw:.4f}  Macro-F1={f1_raw:.4f}')
print(classification_report(y_te, y_pred_raw, target_names=CLASS_NAMES, zero_division=0))

---
## 06 STFT 스펙트로그램 + 2D-CNN 학습
베어링 결함은 특정 주파수에서 주기적으로 나타나므로, 시간-주파수 평면에서 2D 패턴을 학습하면 raw 1D 대비 정확도가 향상되는 것이 일반적이다.

In [ ]:
NPERSEG, NOVERLAP = 128, 64

def to_stft(X1d):
    """(N, 1, window) → (N, 1, F, T) log-magnitude + 윈도우별 z-score."""
    sigs = X1d.squeeze(1)
    _, _, Z = stft(sigs, fs=1.0, nperseg=NPERSEG, noverlap=NOVERLAP, axis=-1)
    mag = np.log1p(np.abs(Z)).astype(np.float32)
    mu = mag.mean(axis=(1,2), keepdims=True)
    sd = mag.std(axis=(1,2),  keepdims=True) + 1e-6
    return ((mag - mu) / sd)[:, None, :, :].astype(np.float32)

X_tr_stft = to_stft(X_tr)
X_va_stft = to_stft(X_va)
X_te_stft = to_stft(X_te)
print(f'STFT 입력 shape: {X_tr_stft.shape}  (B, 1, F={X_tr_stft.shape[2]}, T={X_tr_stft.shape[3]})')

# 클래스별 STFT 예시 시각화
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i in range(3):
    mask = y_te == i
    if not mask.any(): continue
    spec = X_te_stft[mask][0, 0]
    im = axes[i].imshow(spec, aspect='auto', origin='lower', cmap='viridis')
    axes[i].set_title(f'{CLASS_NAMES[i]} STFT 스펙트로그램', fontweight='bold')
    axes[i].set_xlabel('Time frame'); axes[i].set_ylabel('Frequency bin')
    plt.colorbar(im, ax=axes[i], fraction=0.046)
plt.suptitle('STFT 스펙트로그램 (정규화 후)', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
class STFTCNN2D(nn.Module):
    def __init__(self, n_classes=3, dropout=0.3):
        super().__init__()
        self.n_classes = n_classes
        self.conv1 = nn.Conv2d(1, 16, 3, padding=1); self.bn1 = nn.BatchNorm2d(16)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1); self.bn2 = nn.BatchNorm2d(32)
        self.conv3 = nn.Conv2d(32, 64, 3, padding=1); self.bn3 = nn.BatchNorm2d(64)
        self.gap = nn.AdaptiveAvgPool2d((2, 2))
        self.drop = nn.Dropout(dropout)
        self.fc1 = nn.Linear(64*2*2, 64); self.fc2 = nn.Linear(64, n_classes)
    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x))); x = F.max_pool2d(x, 2)
        x = F.relu(self.bn2(self.conv2(x))); x = F.max_pool2d(x, 2)
        x = F.relu(self.bn3(self.conv3(x)))
        x = self.gap(x).flatten(1)
        x = self.drop(F.relu(self.fc1(x)))
        return self.fc2(x)

print('학습 시작 (STFT 2D)...')
model_stft, tr_l2, va_l2, va_a2 = train_cls(STFTCNN2D(n_classes=3),
                                              X_tr_stft, y_tr, X_va_stft, y_va, epochs=30)

model_stft.eval()
with torch.no_grad():
    Xte_st = torch.tensor(X_te_stft).to(device)
    logits_stft = model_stft(Xte_st).cpu().numpy()
y_pred_stft = logits_stft.argmax(1)

acc_stft = accuracy_score(y_te, y_pred_stft)
f1_stft = f1_score(y_te, y_pred_stft, average='macro', zero_division=0)
print(f'\n[STFT 2D Test] Accuracy={acc_stft:.4f}  Macro-F1={f1_stft:.4f}')

---
## 07 raw 1D vs STFT 2D 성능 비교

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# 혼동 행렬 raw
cm_raw = confusion_matrix(y_te, y_pred_raw)
sns.heatmap(cm_raw, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
axes[0].set_title(f'raw 1D-CNN  Acc={acc_raw:.4f}', fontweight='bold')

# 혼동 행렬 STFT
cm_stft = confusion_matrix(y_te, y_pred_stft)
sns.heatmap(cm_stft, annot=True, fmt='d', cmap='Greens', ax=axes[1],
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
axes[1].set_title(f'STFT 2D-CNN  Acc={acc_stft:.4f}', fontweight='bold')

# 모델별 metric 비교 바
metrics = ['Accuracy','Macro-F1','Precision','Recall']
vals_raw  = [acc_raw, f1_raw,
             precision_score(y_te, y_pred_raw,  average='macro', zero_division=0),
             recall_score(y_te, y_pred_raw,     average='macro', zero_division=0)]
vals_stft = [acc_stft, f1_stft,
             precision_score(y_te, y_pred_stft, average='macro', zero_division=0),
             recall_score(y_te, y_pred_stft,    average='macro', zero_division=0)]
x = np.arange(len(metrics)); w = 0.35
axes[2].bar(x - w/2, vals_raw,  w, color=BLUE,  alpha=0.85, label='raw 1D')
axes[2].bar(x + w/2, vals_stft, w, color=GREEN, alpha=0.85, label='STFT 2D')
for i, (a, b) in enumerate(zip(vals_raw, vals_stft)):
    axes[2].text(i - w/2, a + 0.01, f'{a:.3f}', ha='center', fontsize=8)
    axes[2].text(i + w/2, b + 0.01, f'{b:.3f}', ha='center', fontsize=8)
axes[2].set_xticks(x); axes[2].set_xticklabels(metrics)
axes[2].set_title('성능 비교', fontweight='bold'); axes[2].legend()
axes[2].set_ylim(0, 1.1)

plt.suptitle('CWRU 베어링 분류 - raw 1D vs STFT 2D', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

print(f'\n[성능 향상] Accuracy: {acc_raw:.4f} → {acc_stft:.4f}  ({(acc_stft-acc_raw)*100:+.2f}%p)')
print(f'           Macro-F1: {f1_raw:.4f} → {f1_stft:.4f}  ({(f1_stft-f1_raw)*100:+.2f}%p)')

---
## 08 클래스별 상세 성능 및 종합 요약

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
per_cls_raw  = [f1_score(y_te==i, y_pred_raw==i,  zero_division=0) for i in range(3)]
per_cls_stft = [f1_score(y_te==i, y_pred_stft==i, zero_division=0) for i in range(3)]
x = np.arange(3); w = 0.35
ax.bar(x - w/2, per_cls_raw,  w, color=BLUE,  alpha=0.85, label='raw 1D')
ax.bar(x + w/2, per_cls_stft, w, color=GREEN, alpha=0.85, label='STFT 2D')
for i, (a, b) in enumerate(zip(per_cls_raw, per_cls_stft)):
    ax.text(i - w/2, a + 0.01, f'{a:.3f}', ha='center', fontweight='bold')
    ax.text(i + w/2, b + 0.01, f'{b:.3f}', ha='center', fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(CLASS_NAMES)
ax.set_title('클래스별 F1-Score 비교', fontweight='bold')
ax.set_ylim(0, 1.1); ax.legend()
plt.tight_layout(); plt.show()

print('=' * 60)
print('CWRU Bearing 다중 분류 - 최종 요약')
print('=' * 60)
print(f'\n[데이터셋]')
print(f'  유효 파일: {len(files)}개 (IR={class_counts[0]} B={class_counts[1]} OR={class_counts[2]})')
print(f'  Sliding Window: 1024 / Stride 512 / Max {MAX_PER_FILE} per file')
print(f'  파일 단위 split (윈도우 단위 leakage 방지)')
print(f'  Train/Val/Test 윈도우 수: {len(X_tr)}/{len(X_va)}/{len(X_te)}')
print(f'\n[전처리]')
print(f'  - per-window z-score (부하/속도 변화에 강건)')
print(f'  - STFT: nperseg={NPERSEG}, noverlap={NOVERLAP} → ({X_tr_stft.shape[2]}, {X_tr_stft.shape[3]}) 스펙트로그램')
print(f'  - log1p(|Z|) → 윈도우별 z-score (동적 영역 압축)')
print(f'\n[모델]')
print(f'  raw 1D : WDCNN1D (wide kernel=64, stride=8 → 저주파 특징)')
print(f'  STFT 2D: STFTCNN2D (3x3 conv 3층 → 시간-주파수 패턴)')
print(f'\n[성능 비교]')
print(f'  raw 1D  : Acc={acc_raw:.4f}  F1={f1_raw:.4f}')
print(f'  STFT 2D : Acc={acc_stft:.4f}  F1={f1_stft:.4f}')
print(f'  → STFT 향상: Acc {(acc_stft-acc_raw)*100:+.2f}%p, F1 {(f1_stft-f1_raw)*100:+.2f}%p')
print(f'\n[인사이트]')
print(f'  - BPFI/BPFO 결함 주파수가 STFT 평면에서 직접 관찰됨 → 2D 패턴 학습이 유리')
print(f'  - 파일 단위 split은 윈도우 단위 split 대비 성능 수치는 낮아 보이나 신뢰도가 훨씬 높음')
print(f'  - 추가 개선: envelope spectrum, mixup, time-shift augmentation')
print('=' * 60)